# デコーダ：自己回帰・Masked Attention・Cross-Attention

このノートブックでは、Transformer の後半部分である
**デコーダ（Decoder）** を学びます。

書籍 4-13〜4-15 節（図4.50〜4.59）の内容をカバーします。

## 目次
1. デコーダの全体像（図4.50）
2. 自己回帰とは？（図4.51）
3. Output Embedding（図4.52, 4.53）
4. Masked Multi-Head Attention の位置づけ（図4.54, 4.55）
5. Mask のない Attention（復習）
6. なぜ Mask が必要なのか？
7. Mask 行列の仕組み（図4.57）
8. Masked Attention をコードで実装する
9. Mask あり vs なしの比較
10. Masked MHA 以降の処理（図4.58）
11. Cross-Attention（図4.59）
12. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 日本語フォント設定（macOS）
plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False

# デコーダの設定（書籍の例に合わせる）
# デコーダ側は3トークン: [BOS], 富士山, は
n_tokens_dec = 3
d_model = 6
n_heads = 3
d_k = d_model // n_heads  # = 2

# エンコーダ側は7トークン
n_tokens_enc = 7

words_dec = ["[BOS]", "富士山", "は"]
words_enc = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

print("=== デコーダの設定 ===")
print(f"デコーダ側トークン: {words_dec}  ({n_tokens_dec}個)")
print(f"エンコーダ側トークン: {words_enc}  ({n_tokens_enc}個)")
print(f"d_model: {d_model}, ヘッド数: {n_heads}, d_k: {d_k}")

## 1. デコーダの全体像（図4.50）

Transformer は **エンコーダ** と **デコーダ** の2つで構成されています。

```
┌─────────── エンコーダ ×N ───────────┐     ┌──────────── デコーダ ×N ────────────────────┐
│                                     │     │                                             │
│  Input Embedding + PE               │     │  Output Embedding + PE                      │
│       ↓                             │     │       ↓                                     │
│  Multi-Head Attention               │     │  ★ Masked Multi-Head Attention ★           │
│       ↓                             │     │       ↓                                     │
│  Add & Norm                         │     │  Add & Norm                                 │
│       ↓                             │     │       ↓                                     │
│  Feed Forward                       │──→──│  ★ Multi-Head Attention (Cross-Attention) ★ │
│       ↓                             │     │       ↓                                     │
│  Add & Norm                         │     │  Add & Norm                                 │
│                                     │     │       ↓                                     │
└─────────────────────────────────────┘     │  Feed Forward                               │
                                            │       ↓                                     │
                                            │  Add & Norm                                 │
                                            └─────────────────────────────────────────────┘
                                                    ↓
                                               Linear → Softmax → 出力
```

デコーダ特有の処理は **★ で示した2つ** です：

| 処理 | エンコーダとの違い |
|------|-------------------|
| **Masked Multi-Head Attention** | 未来のトークンを見えないようにマスクする |
| **Cross-Attention** | エンコーダの出力を K, V として使う |

それ以外（Add & Norm, Feed Forward）はエンコーダと全く同じです。

## 2. 自己回帰とは？（図4.51）

デコーダの最大の特徴は **自己回帰（Autoregressive）** であることです。

**前の出力を入力として使い、次に出力するトークンを1つずつ決定** していきます。

### 英文和訳の例

入力: "Mount Fuji looks beautiful in spring."

```
ステップ1: [BOS]                         → 富士山
ステップ2: [BOS] 富士山                   → は
ステップ3: [BOS] 富士山 は                 → 春
ステップ4: [BOS] 富士山 は 春              → に
ステップ5: [BOS] 富士山 は 春 に           → 美しい
ステップ6: [BOS] 富士山 は 春 に 美しい     → 。
ステップ7: [BOS] 富士山 は 春 に 美しい 。  → [EOS]
```

### BOS と EOS

| 記号 | 意味 | 役割 |
|------|------|------|
| **[BOS]** | Beginning of Sentence | 「文の開始」を示す特殊トークン。デコーダの最初の入力 |
| **[EOS]** | End of Sentence | 「文の終了」を示す特殊トークン。これが出力されたら生成終了 |

以降の解説では、図4.51の青でハイライトされた部分、
つまり **[BOS], 富士山, は** の3トークンが入力値となっている
ステップに絞って解説します。

In [ ]:
# 図4.51: 自己回帰による出力の流れを可視化

output_tokens = ["[BOS]", "富士山", "は", "春", "に", "美しい", "。", "[EOS]"]

fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

# 各ステップを描画
for step in range(7):
    y = 6 - step * 0.8
    
    # 入力トークン
    input_tokens = output_tokens[:step+1]
    input_text = " ".join(input_tokens)
    
    # 出力トークン
    out_token = output_tokens[step+1]
    
    # ステップ番号
    ax.text(0.02, y, f"Step {step+1}:", fontsize=10, fontweight='bold',
            va='center', fontfamily='monospace')
    
    # 入力
    # ハイライト: step==2 の場合（[BOS], 富士山, は が入力）
    if step == 2:
        bbox = dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
    else:
        bbox = dict(boxstyle='round', facecolor='lightyellow', alpha=0.5)
    ax.text(0.15, y, input_text, fontsize=10, va='center',
            fontfamily='monospace', bbox=bbox)
    
    # 矢印
    ax.annotate('', xy=(0.78, y), xytext=(0.72, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    # 出力
    ax.text(0.82, y, out_token, fontsize=11, va='center',
            fontweight='bold', color='#e74c3c',
            bbox=dict(boxstyle='round', facecolor='#fadbd8', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 7)
ax.set_title('図4.51: デコーダの自己回帰的な出力（前の出力が次の入力になる）',
             fontsize=13, fontweight='bold')
ax.text(0.15, -0.3, '※ 青色のステップ（Step 3）を例に解説していきます',
        fontsize=10, color='blue', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Output Embedding（図4.52, 4.53）

デコーダの入力処理は **エンコーダと同じ** です。

1. **トークン化**: [BOS], 富士山, は → トークン ID（001, 111, 333 など仮の値）
2. **Word Embedding**: トークン ID → 6次元ベクトル
3. **Positional Encoding**: 位置情報を加算

結果は **3×6 の行列** になります（3トークン × 6次元）。

In [ ]:
# デコーダの入力を作成（図4.53）

np.random.seed(77)

# Word Embedding（仮のトークンID → 6次元ベクトル）
token_ids = {"[BOS]": 1, "富士山": 111, "は": 333}
embedding = np.round(np.random.randn(n_tokens_dec, d_model) * 0.5, 2)

# Positional Encoding
def positional_encoding(pos, d_model):
    PE = np.zeros(d_model)
    for i in range(d_model // 2):
        PE[2*i] = np.sin(pos / 10000**(2*i / d_model))
        PE[2*i+1] = np.cos(pos / 10000**(2*i / d_model))
    return PE

PE_dec = np.array([positional_encoding(pos, d_model) for pos in range(n_tokens_dec)])

# 入力 = Embedding + PE
X_dec = np.round(embedding + PE_dec, 3)

print("=== デコーダの入力（図4.53）===")
print(f"トークン: {words_dec}")
print(f"形状: {X_dec.shape}  (3×6)")
print()
for i, word in enumerate(words_dec):
    print(f"  {word:8s}: {X_dec[i]}")
print()
print("ポイント: エンコーダでは 7×6 だったが、デコーダでは 3×6（3トークン）")

## 4. Masked Multi-Head Attention の位置づけ（図4.54, 4.55）

デコーダの最初の処理が **Masked Multi-Head Attention** です。

エンコーダの Multi-Head Attention と **ほぼ同じ** ですが、
1つだけ違いがあります：

```
エンコーダ:  softmax( Q₁K₁ᵀ / √d_k ) × V₁
デコーダ:    softmax( Q₁K₁ᵀ / √d_k + M ) × V₁   ← M（マスク）が追加！
```

### 図4.56: 4ステップのどこに Mask があるか

| ステップ | 処理 | エンコーダとの違い |
|:--------:|------|-------------------|
| 1 | Q, K, V を生成 | 同じ（ただし入力は 3×6）|
| 2 | Q₁K₁ᵀ を計算 | 同じ（結果は 3×3）|
| **3** | **Mask処理 + Softmax** | **★ ここが違う！** |
| 4 | V₁ との積を算出 | 同じ |

## 5. Mask のない Attention（復習）

まず、マスクなしの通常の Attention を復習しましょう。
デコーダの3トークンで計算してみます。

In [ ]:
# Mask なしの Attention（エンコーダと同じ）

def softmax(x):
    """各行に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Head 1 の重み行列を生成
np.random.seed(42)
W_Q1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_K1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)
W_V1 = np.round(np.random.randn(d_model, d_k) * 0.3, 3)

# Q, K, V を生成（Step 1）
Q1 = X_dec @ W_Q1  # (3×6) × (6×2) = (3×2)
K1 = X_dec @ W_K1  # (3×2)
V1 = X_dec @ W_V1  # (3×2)

print("=== Step 1: Q, K, V の生成 ===")
print(f"入力 X の形状: {X_dec.shape}  (3×6)")
print(f"Q₁ の形状: {Q1.shape}  (3×2)")
print(f"K₁ の形状: {K1.shape}  (3×2)")
print(f"V₁ の形状: {V1.shape}  (3×2)")

# Q₁K₁ᵀ / √d_k を計算（Step 2）
QK_scaled = Q1 @ K1.T / np.sqrt(d_k)  # (3×3)
print(f"\n=== Step 2: Q₁K₁ᵀ / √d_k (3×3) ===")
print(np.round(QK_scaled, 3))

# Softmax（マスクなし）
attn_no_mask = softmax(QK_scaled)  # (3×3)
print(f"\n=== Softmax（マスクなし）===")
print(np.round(attn_no_mask, 3))
print()

# 問題点を示す
print("--- 問題点 ---")
print(f'「{words_dec[0]}」は [{attn_no_mask[0,0]:.2f}, {attn_no_mask[0,1]:.2f}, {attn_no_mask[0,2]:.2f}] の重みで注目')
print(f'  → 「{words_dec[1]}」や「{words_dec[2]}」も見えている！（まだ出力されていないのに）')
print(f'「{words_dec[1]}」は [{attn_no_mask[1,0]:.2f}, {attn_no_mask[1,1]:.2f}, {attn_no_mask[1,2]:.2f}] の重みで注目')
print(f'  → 「{words_dec[2]}」も見えている！（まだ出力されていないのに）')

## 6. なぜ Mask が必要なのか？

デコーダは **自己回帰的** に1トークンずつ出力します。

つまり、ある時点でまだ出力されていない **「未来のトークン」** を参照してはいけません。

| トークン | 参照できるもの | 参照してはいけないもの |
|----------|---------------|--------------------|
| **[BOS]** | [BOS] のみ | 富士山、は |
| **富士山** | [BOS]、富士山 | は |
| **は** | [BOS]、富士山、は | （なし）|

各トークンは **自分自身と、自分より前に出現しているトークンの内積しか見えない** ようにマスクします。

これが **Masked Multi-Head Attention** の「Masked」の意味です。

## 7. Mask 行列の仕組み（図4.57）

マスクの仕組みはシンプルです。

Q₁K₁ᵀ/√d_k の計算結果に対して、見てはいけない位置に **$-\infty$** を足します。

$$\text{softmax}\left(\frac{Q_1 K_1^T}{\sqrt{d_k}} + M\right)$$

### Mask 行列 M

```
          [BOS]  富士山    は
[BOS]  [   0     -∞      -∞  ]
富士山  [   0      0      -∞  ]
は      [   0      0       0  ]
```

- **0** → そのまま（見てよい）
- **$-\infty$** → 加算すると $-\infty$ になり、$e^{-\infty} = 0$ なので Softmax 後に 0 になる

### なぜ $-\infty$ を足すと消えるのか？

Softmax は $e^x$ を使います：
- $e^{\text{通常の値}}$ → 正の値（注目される）
- $e^{-\infty} = 0$ → 完全に無視される

In [ ]:
# 図4.57: Mask 行列を作成する

# 上三角に -∞ を入れる（未来のトークンをマスク）
mask = np.zeros((n_tokens_dec, n_tokens_dec))
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        if j > i:  # 未来のトークン（自分より右）
            mask[i, j] = -np.inf

print("=== Mask 行列 M ===")
print(f"形状: {mask.shape}  (3×3)")
print()
print(f"{'':8s}  {words_dec[0]:>8s}  {words_dec[1]:>8s}  {words_dec[2]:>8s}")
for i, word in enumerate(words_dec):
    row = []
    for j in range(n_tokens_dec):
        if mask[i, j] == -np.inf:
            row.append('    -∞')
        else:
            row.append(f'{mask[i,j]:6.0f}')
    print(f"{word:8s}  {'  '.join(row)}")

print()
print("0 = 見てよい（そのまま）")
print("-∞ = 見てはいけない（Softmax後に0になる）")

In [ ]:
# 図4.57: Mask の適用過程を可視化

fig, axes = plt.subplots(1, 4, figsize=(18, 4),
                         gridspec_kw={'width_ratios': [1, 0.3, 1, 1]})

# Q₁K₁ᵀ / √d_k
ax = axes[0]
im = ax.imshow(QK_scaled, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_tokens_dec))
ax.set_xticklabels(words_dec, fontsize=10)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=10)
ax.set_title('Q₁K₁ᵀ / √d_k', fontsize=12, fontweight='bold')
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        ax.text(j, i, f'{QK_scaled[i,j]:.2f}', ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)

# +
ax = axes[1]
ax.text(0.5, 0.5, '+', fontsize=30, ha='center', va='center', fontweight='bold')
ax.axis('off')

# Mask M
ax = axes[2]
# マスク用のカラーマップ（0は白、-∞は黒）
mask_display = np.where(mask == -np.inf, 1, 0)
ax.imshow(mask_display, cmap='Greys', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(n_tokens_dec))
ax.set_xticklabels(words_dec, fontsize=10)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=10)
ax.set_title('Mask M', fontsize=12, fontweight='bold')
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        if mask[i, j] == -np.inf:
            ax.text(j, i, '-∞', ha='center', va='center', fontsize=12, color='white', fontweight='bold')
        else:
            ax.text(j, i, '0', ha='center', va='center', fontsize=12, fontweight='bold')

# Mask 適用後
ax = axes[3]
QK_masked = QK_scaled + mask
# 表示用（-∞ は表示できないので nan にする）
QK_masked_display = np.where(np.isinf(QK_masked), np.nan, QK_masked)
im = ax.imshow(QK_masked_display, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n_tokens_dec))
ax.set_xticklabels(words_dec, fontsize=10)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=10)
ax.set_title('Q₁K₁ᵀ/√d_k + M', fontsize=12, fontweight='bold')
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        if mask[i, j] == -np.inf:
            ax.text(j, i, '-∞', ha='center', va='center', fontsize=11, color='gray')
        else:
            ax.text(j, i, f'{QK_masked[i,j]:.2f}', ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('図4.57: Mask の適用（未来のトークンを -∞ で消す）', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 8. Masked Attention をコードで実装する

Mask を適用した後に Softmax を通して、V₁ を掛けます。

In [ ]:
# Masked Attention の完全な計算

print("=== Step 3: Mask + Softmax ===")
print()

# Q₁K₁ᵀ / √d_k + M
QK_masked = QK_scaled + mask
print("Q₁K₁ᵀ/√d_k + M:")
for i, word in enumerate(words_dec):
    row_str = []
    for j in range(n_tokens_dec):
        if np.isinf(QK_masked[i, j]):
            row_str.append('  -∞  ')
        else:
            row_str.append(f'{QK_masked[i,j]:6.3f}')
    print(f"  {word:8s}: [{', '.join(row_str)}]")

# Softmax を適用
attn_masked = softmax(QK_masked)  # -∞ の部分は e^(-∞) = 0 になる
print(f"\nSoftmax 後のアテンション重み:")
for i, word in enumerate(words_dec):
    row = [f'{attn_masked[i,j]:.3f}' for j in range(n_tokens_dec)]
    print(f"  {word:8s}: [{', '.join(row)}]  合計={attn_masked[i].sum():.4f}")

print(f"\n=== Step 4: V₁ を掛ける ===")
head1_output = attn_masked @ V1  # (3×3) × (3×2) = (3×2)
print(f"アテンション重み (3×3) × V₁ (3×2) = Head 1 出力 (3×2)")
print(f"\nHead 1 出力:")
for i, word in enumerate(words_dec):
    print(f"  {word:8s}: {np.round(head1_output[i], 4)}")

In [ ]:
# Mask の効果を確認: 各トークンが何を見ているか

print("=== Mask の効果 ===")
print()
for i, word in enumerate(words_dec):
    print(f'「{word}」が参照できるトークン:')
    for j in range(n_tokens_dec):
        weight = attn_masked[i, j]
        if weight > 0.001:  # 実質的にゼロでない
            bar = '█' * int(weight * 30)
            print(f'  {words_dec[j]:8s}: {weight:.3f} {bar}')
        else:
            print(f'  {words_dec[j]:8s}: {weight:.3f} ← マスクされて見えない！')
    print()

## 9. Mask あり vs なしの比較

In [ ]:
# Mask あり vs なし の比較ヒートマップ

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# マスクなし（エンコーダ方式）
ax = axes[0]
im = ax.imshow(attn_no_mask, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.6)
ax.set_xticks(range(n_tokens_dec))
ax.set_xticklabels(words_dec, fontsize=11)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=11)
ax.set_title('マスクなし（エンコーダ方式）', fontsize=13, fontweight='bold')
ax.set_xlabel('注目される側', fontsize=10)
ax.set_ylabel('注目する側', fontsize=10)
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        ax.text(j, i, f'{attn_no_mask[i,j]:.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)

# マスクあり（デコーダ方式）
ax = axes[1]
im = ax.imshow(attn_masked, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.6)
ax.set_xticks(range(n_tokens_dec))
ax.set_xticklabels(words_dec, fontsize=11)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=11)
ax.set_title('マスクあり（デコーダ方式）', fontsize=13, fontweight='bold')
ax.set_xlabel('注目される側', fontsize=10)
ax.set_ylabel('注目する側', fontsize=10)
for i in range(n_tokens_dec):
    for j in range(n_tokens_dec):
        val = attn_masked[i, j]
        if val < 0.001:
            ax.text(j, i, '0.00', ha='center', va='center', fontsize=12, color='lightgray')
        else:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Mask あり vs なし：未来のトークンへの注目が 0 になる',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("左: すべてのトークンが互いに見える（エンコーダの Self-Attention）")
print("右: 各トークンは自分以前のトークンしか見えない（デコーダの Masked Attention）")
print("  → 右上が 0.00 = 未来のトークンが完全にマスクされている")

## 10. Masked MHA 以降の処理（図4.58）

Masked Multi-Head Attention の出力後の処理は **エンコーダと全く同じ** です。

```
各ヘッドの出力 (3×2) × 3ヘッド
  ↓ 1. Concat（結合）
(3×6)
  ↓ 2. Linear（W_o を掛ける）
(3×6)
  ↓ 3. Add（元の入力を足す = Skip Connection）
(3×6)
  ↓ 4. Layer Normalization
(3×6)
```

入力値と同じ **3×6 行列** が出力されます。

In [ ]:
# Masked MHA の完全な実装（3ヘッド + Add & Norm）

def masked_multi_head_attention(X, n_heads, d_k, mask):
    """Masked Multi-Head Attention"""
    d_model = X.shape[1]
    head_outputs = []
    
    for h in range(n_heads):
        np.random.seed(42 + h * 10)
        W_Q = np.random.randn(d_model, d_k) * 0.3
        W_K = np.random.randn(d_model, d_k) * 0.3
        W_V = np.random.randn(d_model, d_k) * 0.3
        
        Q = X @ W_Q
        K = X @ W_K
        V = X @ W_V
        
        # Mask 適用
        scores = Q @ K.T / np.sqrt(d_k) + mask  # ← ここが違う！
        attn = softmax(scores)
        head_outputs.append(attn @ V)
    
    # Concat
    concat = np.concatenate(head_outputs, axis=1)
    return concat

def layer_norm(x, epsilon=1e-6):
    """Layer Normalization（簡易版）"""
    mu = np.mean(x, axis=1, keepdims=True)
    sigma = np.std(x, axis=1, keepdims=True)
    return (x - mu) / (sigma + epsilon)

# Masked MHA を実行
mha_concat = masked_multi_head_attention(X_dec, n_heads, d_k, mask)

# Linear (W_o)
np.random.seed(999)
W_o = np.random.randn(d_model, d_model) * 0.3
linear_out = mha_concat @ W_o

# Add & Norm
add_out = linear_out + X_dec
norm_out = layer_norm(add_out)

print("=== Masked MHA + Add & Norm の結果 ===")
print(f"入力:   {X_dec.shape}  (3×6)")
print(f"Concat: {mha_concat.shape}  (3×6)")
print(f"Linear: {linear_out.shape}  (3×6)")
print(f"Add:    {add_out.shape}  (3×6)")
print(f"Norm:   {norm_out.shape}  (3×6)")
print()
print("Masked MHA + Add & Norm の出力:")
for i, word in enumerate(words_dec):
    print(f"  {word:8s}: {np.round(norm_out[i], 3)}")
print()
print("→ この 3×6 の行列が次の Cross-Attention の入力になる")

## 11. Cross-Attention（図4.59, 4.60）

デコーダの2つ目の Multi-Head Attention は **Cross-Attention** と呼ばれます。

### エンコーダの Self-Attention との違い

| | Self-Attention | Cross-Attention |
|---|---|---|
| **Q の出所** | デコーダの出力 | デコーダの出力 |
| **K の出所** | デコーダの出力 | **エンコーダの出力** |
| **V の出所** | デコーダの出力 | **エンコーダの出力** |

### 図4.60: 各ヘッドの行列サイズ

```
デコーダ (3×6) ──→ w_Q ──→ Q₁ (3×2)
                                        ↓
エンコーダ (7×6) ──→ w_K ──→ K₁ (7×2)  → Q₁K₁ᵀ = (3×2)(2×7) = 3×7
                                                    ↓
                                              softmax(3×7) × V₁(7×2) = head₁(3×2)
エンコーダ (7×6) ──→ w_V ──→ V₁ (7×2)
```

**ポイント**: Q は 3×2、K/V は 7×2 と **行数が異なる**！

- QK^T の結果は **3×7**（デコーダの3トークン × エンコーダの7トークン）
- 「日本語の各トークンが、英語のどのトークンに注目するか」を表す
- マスクは **不要**（エンコーダの全トークンを参照してよい）

3ヘッド分を計算して Concat すると (3×6) に戻ります。
その後の Add & Norm、Feed Forward はエンコーダと同様です。

### Cross-Attention の意味

- **Q（デコーダから）**: 「日本語のこのトークンが知りたい情報は何？」
- **K（エンコーダから）**: 「英語の各トークンが持っている情報の索引」
- **V（エンコーダから）**: 「英語の各トークンが持っている実際の情報」

つまり、**デコーダが「知りたいこと」をエンコーダの情報から探す** 処理です。

これにより、入力文（英語）の情報を出力文（日本語）の生成に活用できます。

In [ ]:
# Cross-Attention の実装

# エンコーダの出力（ダミーデータ）
np.random.seed(55)
encoder_output = np.round(np.random.randn(n_tokens_enc, d_model) * 0.5, 3)

print("=== Cross-Attention ===")
print(f"Q の元: デコーダの出力 {norm_out.shape}  (3×6)")
print(f"K の元: エンコーダの出力 {encoder_output.shape}  (7×6)")
print(f"V の元: エンコーダの出力 {encoder_output.shape}  (7×6)")
print()

# Head 1 で計算
np.random.seed(300)
W_Q_cross = np.random.randn(d_model, d_k) * 0.3
W_K_cross = np.random.randn(d_model, d_k) * 0.3
W_V_cross = np.random.randn(d_model, d_k) * 0.3

# Q はデコーダから、K と V はエンコーダから
Q_cross = norm_out @ W_Q_cross        # (3×6) × (6×2) = (3×2)
K_cross = encoder_output @ W_K_cross  # (7×6) × (6×2) = (7×2)
V_cross = encoder_output @ W_V_cross  # (7×6) × (6×2) = (7×2)

print(f"Q (デコーダ由来): {Q_cross.shape}  (3×2)")
print(f"K (エンコーダ由来): {K_cross.shape}  (7×2)")
print(f"V (エンコーダ由来): {V_cross.shape}  (7×2)")
print()

# QKᵀ を計算（マスクなし！）
cross_scores = Q_cross @ K_cross.T / np.sqrt(d_k)  # (3×2) × (2×7) = (3×7)
cross_attn = softmax(cross_scores)

print(f"アテンションスコア (QKᵀ/√d_k): {cross_scores.shape}  (3×7)")
print(f"  → デコーダの3トークンがエンコーダの7トークンにどれだけ注目するか")
print()

# 出力
cross_output = cross_attn @ V_cross  # (3×7) × (7×2) = (3×2)
print(f"Cross-Attention の出力: {cross_output.shape}  (3×2)")

In [ ]:
# Cross-Attention のアテンション重みを可視化

fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(cross_attn, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.3)
ax.set_xticks(range(n_tokens_enc))
ax.set_xticklabels(words_enc, fontsize=11, rotation=45)
ax.set_yticks(range(n_tokens_dec))
ax.set_yticklabels(words_dec, fontsize=11)
ax.set_xlabel('エンコーダ側（注目される側）', fontsize=11)
ax.set_ylabel('デコーダ側（注目する側）', fontsize=11)
ax.set_title('Cross-Attention: デコーダがエンコーダのどこに注目しているか',
             fontsize=13, fontweight='bold')

for i in range(n_tokens_dec):
    for j in range(n_tokens_enc):
        ax.text(j, i, f'{cross_attn[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("このヒートマップの見方:")
print("  行: デコーダのトークン（日本語側）")
print("  列: エンコーダのトークン（英語側）")
print("  値が大きい = そのトークンの情報を多く取り込んでいる")
print()
print("例: 「富士山」が 'Fuji' や 'Mount' に高い注目度を示せば、")
print("    翻訳として正しい対応関係を学んでいることになる")

## 12. まとめ

| ポイント | 内容 |
|----------|------|
| **デコーダ** | Transformer の後半。自己回帰的にトークンを1つずつ生成する |
| **自己回帰** | 前の出力を次の入力に使う。[BOS] → 富士山 → は → ... → [EOS] |
| **[BOS] / [EOS]** | 文の開始/終了を示す特殊トークン |
| **Output Embedding** | エンコーダと同じ（Word Embedding + PE）|
| **Masked MHA** | 未来のトークンを $-\infty$ でマスクし、Softmax で 0 にする |
| **Mask 行列 M** | 上三角が $-\infty$、下三角と対角が 0 |
| **Cross-Attention** | Q=デコーダ(3×2)、K/V=エンコーダ(7×2)。QKᵀ は 3×7 |
| **Cross-Attention の意味** | デコーダが入力文の情報から必要な情報を探す |
| **エンコーダと共通** | Add & Norm, Feed Forward は同じ処理 |

### デコーダ1層の処理フロー

```
デコーダ入力 (3×6)
  ├──────── Skip Connection ──────────┐
  ↓                                   │
  Masked Multi-Head Attention          │
  ↓                                   │
  Add & Norm ←────────────────────────┘
  │
  ├──────── Skip Connection ──────────┐
  ↓                                   │
  Cross-Attention ←── エンコーダの出力  │
  ↓   (Q=デコーダ, K/V=エンコーダ)      │
  Add & Norm ←────────────────────────┘
  │
  ├──────── Skip Connection ──────────┐
  ↓                                   │
  Feed Forward                        │
  ↓                                   │
  Add & Norm ←────────────────────────┘
  ↓
  デコーダ出力 (3×6)
```

## 次のステップ

次のノートブックでは、デコーダの最終段階である
**Linear + Softmax** による出力トークンの予測を学びます。

デコーダ出力 (3×6) に W_out (6×10,000) を掛けて
10,000個の候補から最も確率の高いトークンを選択します。